# 06 · End-to-end pipeline

`src/pipeline.py` chains every stage — load → clean → features → forecast → supervised models → report — reusing the exact modules notebooks 01-05 use. This notebook just runs it and renders the result.

```bash
python -m src.pipeline            # from EY Training/ml_superstore/
```

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from IPython.display import Markdown
from src.pipeline import run_pipeline
from src.config import TABLE_DIR, OUTPUT_DIR

## Run

In [2]:
report = run_pipeline(quiet=False)
print('\nkeys:', list(report))

[20:12:53] load + clean
[20:12:53] feature engineering


[20:12:54] forecasting


[20:12:54] supervised models


[20:13:07] done in 13.7s -> pipeline_report.json, REPORT.md

keys: ['generated_utc', 'source_csv', 'n_order_lines', 'date_range', 'features', 'forecast', 'models', 'runtime_sec']


## Rendered report

In [3]:
Markdown((OUTPUT_DIR / 'REPORT.md').read_text(encoding='utf-8'))

# Superstore ML — pipeline report

*Generated 2026-09-01T14:42:53+00:00 · runtime 13.7s*

- **Source**: `Sample - Superstore.csv`
- **Order lines**: 9,994  (2014-01-03 → 2017-12-30)
- **Engineered features**: 26 · customers 793

## Forecast (monthly sales)

- Holt-Winters hold-out MAE **11,456** (MAPE 22.6%)
- SARIMA hold-out MAE 13,411
- Holt-Winters rolling-backtest MAE 12,314
- Next 12 months projected **914,903** vs last full year 733,215 (**+24.8%**)

## Supervised models (train ≤2016 / test 2017)

- Best profit regressor: **RandomForest** — MAE 21.4, RMSE 139.2, R² 0.668
- Best loss classifier: **RandomForest** — ROC-AUC 0.988, PR-AUC 0.956, F1 0.865 (tuned threshold 0.50 → F1 0.865)

Artefacts: `outputs/tables/pipeline_*.csv`, `outputs/tables/pipeline_report.json`, `outputs/models/pipeline_*.joblib`.


## Forecast table

In [4]:
pd.read_csv(TABLE_DIR / 'pipeline_forecast.csv', index_col=0).round(0)

,holt_winters,sarima_mean,lo80,hi80
2018-01-01,49691.0,52929.0,33806.0,72052.0
2018-02-01,41838.0,35122.0,15999.0,54245.0
2018-03-01,74869.0,71213.0,52042.0,90384.0
2018-04-01,61536.0,51317.0,32104.0,70531.0
2018-05-01,68506.0,61811.0,42555.0,81067.0
2018-06-01,65392.0,63879.0,44580.0,83177.0
2018-07-01,67082.0,57902.0,38561.0,77242.0
2018-08-01,66665.0,68938.0,49555.0,88320.0
2018-09-01,108144.0,98286.0,78862.0,117711.0
2018-10-01,77814.0,87244.0,67777.0,106710.0


## Model scoreboards

In [5]:
reg = pd.DataFrame(report['models']['regression']).T
clf = pd.DataFrame(report['models']['classification']).T
display(reg.round(3))
display(clf.round(4))

,MAE,RMSE,R2
Dummy(mean),64.684,241.829,-0.000
Ridge,58.661,183.108,0.427
RandomForest,21.425,139.241,0.668
GradBoost,25.539,113.420,0.780


,ROC_AUC,PR_AUC,F1,threshold
Dummy(prior),0.5000,0.1872,0.0000,0.5
LogReg,0.9846,0.9449,0.8278,0.5
RandomForest,0.9881,0.9559,0.8649,0.5
GradBoost,0.9861,0.9466,0.8038,0.5


## Raw report JSON

In [6]:
print(json.dumps(report, indent=2)[:2500], '...')

{
  "generated_utc": "2026-09-01T14:42:53+00:00",
  "source_csv": "Sample - Superstore.csv",
  "n_order_lines": 9994,
  "date_range": [
    "2014-01-03",
    "2017-12-30"
  ],
  "features": {
    "n_rows": 9994,
    "n_engineered_features": 26,
    "engineered_features": [
      "profit_margin",
      "is_loss",
      "order_year",
      "order_month",
      "order_quarter",
      "order_week",
      "order_dayofweek",
      "order_dayofyear",
      "order_is_weekend",
      "order_is_month_end",
      "order_is_quarter_end",
      "month_sin",
      "month_cos",
      "dow_sin",
      "dow_cos",
      "price_per_unit",
      "log_sales",
      "is_discounted",
      "discount_bucket",
      "ship_days",
      "cust_prior_lines",
      "cust_prior_sales_sum",
      "cust_prior_sales_mean",
      "cust_prior_loss_rate",
      "cust_days_since_first",
      "cust_is_first_order"
    ],
    "n_customers": 793
  },
  "forecast": {
    "holdout": {
      "HoltWinters": {
        "MAE": 1145

## What ships

| stage | module | notebook | key output |
|---|---|---|---|
| clean | `data_loader` | 01 | typed frame, DQ report |
| EDA | — | 02 | 9 figures, discount→loss story |
| features | `features` | 03 | 9994×32 matrix, RFM, leakage-safe history |
| forecast | `forecasting` | 04 | Holt-Winters/SARIMA, 12-month projection |
| models | `modeling` | 05 | profit regressor + loss classifier |
| glue | `pipeline` | 06 | `REPORT.md` + `pipeline_report.json` |

Re-run everything with `python -m src.pipeline`; re-execute the notebooks with the `jupyter nbconvert --execute` loop in the README.